In [ ]:
import os
os.environ["XLA_FLAGS"] = '--xla_force_host_platform_device_count=8'

In [ ]:
from jax import numpy as jnp
from jax import lax, jit, make_jaxpr
import jax

import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(8)

In [ ]:
jax.device_count()

In [ ]:
jax.config.update("jax_enable_x64", True)

In [ ]:
from jax.scipy.stats import gamma as jaxgamma

def get_gamma_densities(window_len, mean, sd):
    var = sd ** 2.0
    scale = var / mean
    a = mean / scale
    cum_dens = jaxgamma.cdf(jnp.arange(window_len + 1), a=a,scale=scale)
    diff_series = jnp.diff(cum_dens)
    return diff_series/diff_series.sum()


In [ ]:
from numpyro import distributions as dist
from jax.random import PRNGKey, split

In [ ]:
spops = np.ones((16,16)) * 1000.0

In [ ]:
window_len = 32

In [ ]:
from scipy import sparse


ssm = sparse.coo_array((256,256))
#jax.experimental.sparse.BCOO.from_scipy_sparse(ssm)

In [ ]:
ssm[0,0]

In [ ]:
def gen_neighbours(n):
    nhood_idx = np.arange(n*n)
    nhood_spatial = nhood_idx.reshape(n,n)
    nhood_mat = np.zeros((n*n,n*n))
    for xi in range(n):
        for yi in range(n):
            nhood_i = xi*n + yi
            nhood_mat[nhood_i,nhood_i] = 1.0
            if xi > 0:
                upper = nhood_i - n
                nhood_mat[nhood_i,upper] = 0.1
            if xi < n-1:
                lower = nhood_i + n
                nhood_mat[nhood_i,lower] = 0.1
            if yi > 0:
                left = nhood_i - 1
                nhood_mat[nhood_i, left] = 0.1
            if yi < n-1:
                right = nhood_i + 1
                nhood_mat[nhood_i, right] = 0.1
    return nhood_mat

In [ ]:
def gen_neighbours_sparse(n):
    nhood_idx = np.arange(n*n)
    nhood_spatial = nhood_idx.reshape(n,n)
    #nhood_mat = np.zeros((n*n,n*n))
    row = []
    col = []
    data = []

    def add_point(xi,yi,d):
        row.append(xi)
        col.append(yi)
        data.append(d)

    for xi in range(n):
        for yi in range(n):
            nhood_i = xi*n + yi

            add_point(nhood_i,nhood_i,1.0)
            if xi > 0:
                upper = nhood_i - n
                add_point(nhood_i,upper,0.1)
                #nhood_mat[nhood_i,upper] = 0.1
            if xi < n-1:
                lower = nhood_i + n
                add_point(nhood_i,lower,0.1)
                #nhood_mat[nhood_i,lower] = 0.1
            if yi > 0:
                left = nhood_i - 1
                add_point(nhood_i,left,0.1)
                #nhood_mat[nhood_i, left] = 0.1
            if yi < n-1:
                right = nhood_i + 1
                add_point(nhood_i,right,0.1)
                #nhood_mat[nhood_i, right] = 0.1
    row = np.array(row)
    col = np.array(col)
    data = np.array(data)
    return sparse.coo_array((data, (row, col)), shape=(n*n, n*n))

In [ ]:
def gen_neighbours_sparsev(n):
    nhood_idx = np.arange(n*n)
    nhood_spatial = nhood_idx.reshape(n,n)
    #nhood_mat = np.zeros((n*n,n*n))
    row = []
    col = []
    data = []

    def point_idx(x,y):
        return x*n + y

    def _add_point(xi,yi,d):
        row.append(xi)
        col.append(yi)
        data.append(d)

    def add_point(poss, posd, d):
        if validate(posd) and validate(poss):
            pidxs = point_idx(*poss)
            pidxd = point_idx(*posd)
            _add_point(pidxs, pidxd, d)

    def validate(point):
        return (point>=0).all() and (point<n).all()

    for xi in range(n):
        for yi in range(n):
            nhood_i = xi*n + yi
            point = np.array((xi,yi))

            def add(offset,value):
                add_point(point,point+np.array(offset),value)

            add_point(point,point,1.0)

            add((0,-1),0.2)
            add((0,1),0.2)
            add((1,0),0.2)
            add((-1,0),0.2)
            add((-1,-1),0.17)
            add((-1,1),0.17)
            add((1,1),0.17)
            add((1,-1),0.17)

    row = np.array(row)
    col = np.array(col)
    data = np.array(data)
    return sparse.coo_array((data, (row, col)), shape=(n*n, n*n))

In [ ]:
NX = 16
NY = 16

pt = np.array((8,15),dtype = int)
offset = np.array((-2,2),dtype=int)

pt + offset

In [ ]:
N = 32

nhm = gen_neighbours(N)
nhms = gen_neighbours_sparse(N)
nhmsv = gen_neighbours_sparsev(N)

In [ ]:
nhm = nhmsv

In [ ]:
spops = np.ones(N*N)
full_spops = spops.copy()
ipops = np.zeros(N*N)
rpops = ipops.copy()
ipops[3*N+3] = 0.01
ipops[15*N+12] = 0.05

ipops[25*N+5] = 0.2

step_spops = [spops.reshape(N,N)]
step_ipops = [ipops.reshape(N,N)]

rrate = 0.1
crate = 0.3
wrate = 0.03

for i in range(190):
    newi = nhms@ipops * (spops / full_spops) * crate
    rnew = ipops * rrate
    wnew = rpops * wrate
    ipops = ipops + newi - rnew
    spops = spops - newi + wnew
    spops = np.maximum(spops, np.zeros_like(spops))
    rpops = rpops + rnew - wnew
    
    step_spops.append(spops.reshape(N,N))
    step_ipops.append(ipops.reshape(N,N))

In [ ]:
spt = np.array(step_spops)
ipt = np.array(step_ipops)


In [ ]:
from plotly import express as px

In [ ]:
fig = px.imshow(ipt, animation_frame=0, zmin=0.0,zmax=1.0)
fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 120
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 60
fig

In [ ]:
pd.Series(ipt[:,3,3]).plot()

In [ ]:
pops = pops/pops.sum()

In [ ]:
scomps = np.prod(spops.shape)

In [ ]:
mm = np.zeros(scomps,scomps)

In [ ]:
mm = np.diag(np.ones(scomps),0) + np.diag(np.ones(scomps-1)*0.1,1) + np.diag(np.ones(scomps-1)*0.1,-1)

In [ ]:
mm @ spops.flat

In [ ]:
def renew_spatial(gmean, gsd, rt, wperiod, mm, k=None):
    densities = get_gamma_densities(window_len, gmean, gsd)

    wrate = 1.0/wperiod

    if k is None:
        stochastic = False
    else:
        stochastic = True
        ks = split(k, ntimes)

    def state_update(state, t):
        target_inc = mm @ (densities * state["incidence"]).sum(axis=1) # Calculated incidence
        target_inc *= state["suscept"] / isus
        target_inc *= rt[t]
        if stochastic:
            target_inc = dist.Poisson(target_inc).sample(ks[t])
        actual_inc = jnp.minimum(target_inc, state["suscept"])  # Incidence after ceiling applied
        waned = state["recovered"] * wrate
        suscept = state["suscept"] - actual_inc + waned # Susceptible depletion
        recovered = state["recovered"] - waned + actual_inc
        inc = jnp.concat([jnp.array([actual_inc]).T, state["incidence"][:,:-1]],axis=1)  # Move up in matrix
        out = {"incidence": actual_inc, "suscept": suscept}
        return {"suscept": suscept, "recovered": recovered, "incidence": inc}, out

    end_state, outputs = lax.scan(state_update, istate, jnp.arange(ntimes))
    #full_inc = jnp.concatenate([init_incidence, jnp.array(outputs["incidence"])])
    #return RenewResults(outputs, full_incidence=full_inc)
    return end_state, outputs


In [ ]:
def npmodel():

    m0 = numpyro.sample("m0", dist.TruncatedNormal(1.2, 0.1, low=0.5, high=2.0))
    m1 = numpyro.sample("m1", dist.TruncatedNormal(0.01, 0.01, low=0.0, high=0.1))
    m2 = numpyro.sample("m2", dist.TruncatedNormal(0.02, 0.01, low=0.0, high=0.1))
    m3 = numpyro.sample("m3", dist.TruncatedNormal(1.4, 0.1, low=0.5, high=2.0))

    mm = jnp.array([
        [m0,m1],
        [m2,m3]
    ])

    gmean = numpyro.sample("gmean", dist.TruncatedNormal(9.5,2.0, low=4.0, high=20.0))
    gsd = numpyro.sample("gsd", dist.Uniform(1.0, 6.0))# dist.TruncatedNormal(3.0,0.1, low=1.5, high=6.0))
    wperiod = numpyro.sample("wperiod", dist.TruncatedNormal(100.0,20.0, low=40.0, high=300.0))

    fstate, out = renew(gmean, gsd, rtactual, wperiod, mm)
    inc = out["incidence"]#.sum(axis=1)

    numpyro.factor("lp", dist.Normal(jnp.log(inc+1e-32), 0.1).log_prob(jnp.log(target+1e-32)).mean() * 20.0)

In [ ]:
def npmodel():

    m0 = numpyro.sample("m0", dist.TruncatedNormal(1.2, 0.1, low=0.5, high=2.0))
    m1 = numpyro.sample("m1", dist.TruncatedNormal(0.01, 0.01, low=0.0, high=0.1))
    m2 = numpyro.sample("m2", dist.TruncatedNormal(0.02, 0.01, low=0.0, high=0.1))
    m3 = numpyro.sample("m3", dist.TruncatedNormal(1.4, 0.1, low=0.5, high=2.0))

    mm = jnp.array([
        [m0,m1],
        [m2,m3]
    ])

    gmean = numpyro.sample("gmean", dist.TruncatedNormal(9.5,2.0, low=4.0, high=20.0))
    gsd = numpyro.sample("gsd", dist.Uniform(1.0, 6.0))# dist.TruncatedNormal(3.0,0.1, low=1.5, high=6.0))
    wperiod = numpyro.sample("wperiod", dist.TruncatedNormal(100.0,20.0, low=40.0, high=300.0))

    fstate, out = renew(gmean, gsd, rtactual, wperiod, mm)
    inc = out["incidence"]#.sum(axis=1)

    numpyro.factor("lp", dist.Normal(jnp.log(inc+1e-32), 0.1).log_prob(jnp.log(target+1e-32)).mean() * 20.0)